# 1. Introduction
In this notebook we'll start from a time series of Sentinel-2 data over an agricultural area in Malawi, located in between the towns of Nkhotakota in the east and Kasungu in the west. 

We will go through the different steps of pre-processing optical imagery, i.e.:
- masking clouds based on the Sentinel-2 scene classification layer
- 10-daily compositing, or in other words making sure we have one observation every 10 days
- interpolation of no data to make sure we have a continuous time series without any gaps

In the process, you have the chance to get acquainted with het NetCDF file format (nowadays commonly used to store geospatial raster data) and the tools in Python to handle these files (most notably the xarray package).

# 2. Preparations

<div class="alert alert-block alert-warning">
<b>Running this notebook on CDSE notebooks?</b><br>

Make sure you select the **Geo science** python kernel.<br>

Execute the following cell to install some required python packages.

</div>

In [ ]:
!pip install scikit-image numba --quiet

Now we import the necessary functions that will be used in this notebook.

If you want to learn what happens exactly in these functions, please browse to the folder `vito_agri_tutorials` and locate the specific function you're looking for in this folder.<br>
You can find out the location using the relative paths as specified below, for instance:<br>
`interpolate_ts` function is located in vito_agri_tutorials > utils > interpolate.py

In [ ]:
# Import the necessary python libraries...
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# import custom python functions all defined in separate .py files
from vito_agri_tutorials.utils.mask import mask_ts
from vito_agri_tutorials.utils.composite import composite_ts
from vito_agri_tutorials.utils.interpolate import interpolate_ts

In the next cell, we specify the folder where the data is located we'll be working with in this exercise.<br>
We encourage you to open this `data` folder using the file explorer on the left hand side of your screen and have a look which files are located there!

In [ ]:
# define folder where to find the data for this exercise
indir = Path('./data')

# 3. Demo on 20 m data

Let's first have a look at the 20m Sentinel-2 bands, which are located in a separate file.<br>
Please note the data has already been resampled to 10m before, to match the 10 m bands located in another file.

In the next cell, we specify which file to open and open the file using xarray (function `open_dataset`):

In [ ]:
infile = str(indir / 'S2_L2A_Malawi_20m_small.nc')
ds = xr.open_dataset(infile)
ds

So in this file there are 6 variables and each variable consists of a 3-dimensional array (time, x, y).<br>
There are 60 available dates in the time series and we have a total of 200 x 200 pixels available.<br>
Let's plot some data to get a sense on how the data looks like...<br>
Here, we plot band 11, i.e. the first of two SWIR bands in Sentinel-2.

In [ ]:
# Plot the first SWIR band (B11) for one pixel:
fig, ax = plt.subplots()
# in the next line of code, we select B11,
# convert it to a data array and then extract
# all its values for pixel located at position x=10, y=10
b11 = ds[['B11']].to_array().values[0, :, 10, 10]
# extract the timestamps
time = ds.coords['timestamp'].values
# plot B11 vs time
ax.plot(time, b11, '-o')
plt.xticks(rotation=45, ha='right')
plt.title('B11 original')
plt.show()

As you can tell from the figure above, the signal looks quite messy and requires some further cleaning before proceeding with any analysis.

First step will be cloud masking. For this, we need the scene classification band, which is available in the file we just opened.

In [ ]:
# We convert it to a data array for further processing...
scl_20 = ds[['SCENECLASSIFICATION']].to_array()

# within the scene classification layer, the following values point
# to clouds/shadows:
scl_mask_values = [1, 3, 8, 9, 10, 11]

# create mask (True = not to be masked, False = to be masked)
mask = np.logical_not(scl_20.isin(scl_mask_values))
mask.attrs = scl_20.attrs.copy()

# NOTE that the quality of the scene classification layer for sentinel-2 is not always optimal.
# Several enhancements are possible before applying this layer for masking. An example can be found here:
# https://github.com/dzanaga/satio-pc/blob/main/satio_pc/preprocessing/clouds.py#L60
# This is however beyond the scope of this exercise.

# Get the 20 m bands from the file
ts_20 = ds[['B05', 'B06', 'B07', 'B11', 'B12']].to_array()

# Apply the mask to the data
ts_20_masked = mask_ts(ts_20, mask)

# Plot the result for the same pixel plotted earlier.
fig, ax = plt.subplots()
b11 = ts_20_masked.sel(variable='B11').values[:, 10, 10]
time = ds.coords['timestamp'].values
ax.plot(time, b11, '-o')
plt.xticks(rotation=45, ha='right')
plt.title('B11 masked')
plt.show()

Compare this plot with the earlier plot. What changed?

Now that we got rid of (most) clouds, let's proceed with the next step.<br>
Now we create a 10-daily summary of the data through temporal compositing.<br>
The result of this step will be that you end up with a time series having one observation every 10 days.<br>
Each observation is the median of all available observations within the 10 day window.<br>
This makes the time series more uniform (number of available observations highly depends on the location!)

In [ ]:
# apply compositing to the timeseries
# (have a look in composite.py to learn what happens)
ts_20_comp = composite_ts(ts_20_masked, freq=10, window=20, mode='median')

# plot the result for our pixel:
fig, ax = plt.subplots()
b11 = ts_20_comp.sel(variable='B11').values[:, 10, 10]
time = ts_20_comp.coords['timestamp'].values
ax.plot(time, b11, '-o')
plt.xticks(rotation=45, ha='right')
plt.title('B11 composited')
plt.show()

Again, compare this one with the previous graph and reflect on what happened.

As a final step, we linearly interpolate any remaining gaps in the time series (for some 10 day windows, there was not a single valid observation as can be seen as interruptions of the graph above).

In [ ]:
# apply interpolation to the time series
# (have a look in interpolate.py to learn what happens in the background)
ts_20_fin = interpolate_ts(ts_20_comp)

# plot the result for our pixel:
fig, ax = plt.subplots()
b11 = ts_20_fin.sel(variable='B11').values[:, 10, 10]
time = ts_20_comp.coords['timestamp'].values
ax.plot(time, b11, '-o')
plt.xticks(rotation=45, ha='right')
plt.title('B11 interpolated')
plt.show()

This is the result of the pre-processing to ensure smooth, un-interrupted time series.

# 4. Do it yourself: pre-process the 10m bands

Now it's time to move to the 10 m resolution data.<br>

In the following parts, you will need to enter some code yourself.<br>
Whenever you see *< YOUR CODE HERE >*, please replace this statement with some code based on the example of the 20 m data.

Open the associated file and go through each of the pre-processing steps as explained above:

In [ ]:
# Select the correct data file and open it
infile = < YOUR CODE HERE >
ds = xr.open_dataset(infile)
ds

In [ ]:
# Get the 10 m bands from the file
ts_10 = ds[[ < YOUR CODE HERE > ]].to_array()

# Apply the mask to the data
ts_10_masked = < YOUR CODE HERE > 

# Apply temporal compositing
ts_10_comp = < YOUR CODE HERE > 

# Apply linear interpolation
ts_10_fin = < YOUR CODE HERE > 
ts_10_fin

# plot comparison between original NIR band and the result of pre-processing
fig, ax = plt.subplots()
b08 = ts_10.sel(variable='B08').values[:, 10, 10]
time = ds.coords['timestamp'].values
ax.plot(time, b08, '-o')
time = ts_10_fin.coords['timestamp'].values
b08 = ts_10_fin.sel(variable='B08').values[:, 10, 10]
ax.plot(time, b08, '-or')
plt.xticks(rotation=45, ha='right')
plt.title('NIR band pre-processing')
plt.show()


In [ ]:
# As a final step, we merge all bands together
ts_fin = xr.concat([ts_10_fin, ts_20_fin], dim='variable')
ts_fin

In [ ]:
# And we free up some memory before proceeding...
del ts_20, ts_20_fin, ts_20_comp, ts_20_masked
del ts_10, ts_10_fin, ts_10_comp, ts_10_masked

# 5. Computation of vegetation indices
Now that we have our data ready to go, let's compute a well-known vegetation index used as a basis for many agricultural monitoring applications, i.e. the NDVI.

In [ ]:
# Compute NDVI
# isolate the NIR band (B08)
b08 = ts_fin.sel(variable='B08').values
# isolate the RED band (B04)
b04 = ts_fin.sel(variable='B04').values
# compute the index
ndvi = (b08 - b04) / (b08 + b04)
# Display the shape of the result
ndvi.shape

Note that the result is a 3D matrix with 28 time steps and 200 x 200 pixels.<br>
Recall originally our data contained 60 time steps. How come we moved to 28?

In [ ]:
# Visualize spatially for one particular date
# (we select the 11th available date)
fig, ax = plt.subplots()
ndviplot = plt.imshow(ndvi[10, ...])
fig.colorbar(ndviplot, ax=ax)
plt.show()

In [ ]:
# Visualize temporally for one pixel
# retrieve the time coordinate
time = ts_fin.coords['timestamp'].values

# now plot the time series for our pixel located at position 10,10
fig, ax = plt.subplots()
ax.plot(time, ndvi[:, 10, 10], '-or')
plt.xticks(rotation=45, ha='right')
plt.title('NDVI for pixel (10,10)')
plt.show()


END OF THE EXERCISE

Please note that pre-processing will be repeated in the next exercise, where you will be able to find the solution of this exercise.